In [1]:
import numpy as np
import os
import pandas as pd
import re
from pathlib import Path

In [2]:
import pyreadr

In [3]:
print(os.getcwd())

/users/antoumos/benchmark


In [4]:
result = pyreadr.read_r("/store_new/mch/msclim/antoumos/R/develop/NOWPRECIP/new_project/radar_data/obs.new.220040700.rda")

In [5]:
import numpy as np

da = result["obs"]

print("min:", float(da.min(skipna=True)))
print("max:", float(da.max(skipna=True)))
print("mean:", float(da.mean(skipna=True)))
print("std:", float(da.std(skipna=True)))

qs = da.quantile([0, 0.01, 0.1, 0.5, 0.9, 0.99, 1.0], skipna=True)
print(qs)

min: 0.0
max: 1.7204732762442694
mean: 0.025912389260554395
std: 0.08981992061963555
<xarray.DataArray (quantile: 7)> Size: 56B
array([0.        , 0.        , 0.        , 0.        , 0.06550781,
       0.47773685, 1.72047328])
Coordinates:
  * quantile  (quantile) float64 56B 0.0 0.01 0.1 0.5 0.9 0.99 1.0


In [6]:
from __future__ import annotations

import numpy as np
import pandas as pd
import xarray as xr


def season_from_month(month: int) -> str:
    # DJF/MAM/JJA/SON
    if month in (12, 1, 2):
        return "DJF"
    if month in (3, 4, 5):
        return "MAM"
    if month in (6, 7, 8):
        return "JJA"
    return "SON"

def expand_event_to_samples_np(arr, event_id, lead_steps=3, hist_steps=3, step=1):
    if arr.ndim != 3:
        raise ValueError(f"Expected (H,W,T), got {arr.shape}")
    
    ### want to craate training pairs of X = [t0-20,t0 -10,t0], Y = [t0+30]
    
    H, W, T = arr.shape
    last = T - 1

    t0_min = hist_steps - 1          # 2 minimum t0 to contain -10,-20 steps
    t0_max = last - lead_steps       # I want +30min prediction

    if t0_max < t0_min:
        return (
            np.empty((0, hist_steps, H, W), dtype=np.float32),
            np.empty((0, 1, H, W), dtype=np.float32),
            pd.DataFrame(columns=["event_id", "t0_idx", "target_idx"])
        )

    t0s = np.arange(t0_min, t0_max + 1, step, dtype=int) # get all the t0
    N = len(t0s)
    print(N)

    X = np.empty((N, hist_steps, H, W), dtype=np.float32) # 3 steps for motion training
    Y = np.empty((N, 1, H, W), dtype=np.float32) # target is +30min

    rows = []
    #sliding window
    for i, t0 in enumerate(t0s):
       x = arr[:, :, (t0 - (hist_steps - 1)):(t0 + 1)].to_numpy()  # or .values
       y = arr[:, :, (t0 + lead_steps)].to_numpy()

       X[i] = np.transpose(x, (2, 0, 1)).astype(np.float32, copy=False)
       Y[i, 0] = y.astype(np.float32, copy=False)

    # store metadata
    meta = pd.DataFrame({
        "event_id": event_id,
        "t0_idx": t0s,
        "target_idx": t0s + lead_steps,
    })

    #meta = pd.DataFrame(rows)
    return X, Y, meta



In [7]:
# Pick the observations array #

arr = result["obs"]

# Infer event_id from filename
path = "/store_new/mch/msclim/antoumos/R/develop/NOWPRECIP/new_project/obs.new.220040700.rda"
m = re.search(r"obs\.new\.(.+?)\.rda$", path)
event_id = m.group(1) if m else "unknown"

# Expand
X, Y, meta = expand_event_to_samples_np(arr, event_id=event_id, lead_steps=3, hist_steps=3, step=1)

print("Expanded samples:")
print("X:", X.shape)   # (N, 3, H, W)
print("Y:", Y.shape)   # (N, 1, H, W)
print(meta.head())

# # Sanity: last indices
if len(meta) > 0:
     print("t0 range:", meta.t0_idx.min(), "..", meta.t0_idx.max())
     print("target max:", meta.target_idx.max(), " / last index:", arr.shape[2] - 1)

34
Expanded samples:
X: (34, 3, 501, 371)
Y: (34, 1, 501, 371)
    event_id  t0_idx  target_idx
0  220040700       2           5
1  220040700       3           6
2  220040700       4           7
3  220040700       5           8
4  220040700       6           9
t0 range: 2 .. 35
target max: 38  / last index: 38


In [8]:
print(meta.head(3))
print(meta.tail(3))
print(meta["t0_idx"].describe())
print(meta["t0_idx"].unique()[:10], " ... n_unique =", meta["t0_idx"].nunique())

    event_id  t0_idx  target_idx
0  220040700       2           5
1  220040700       3           6
2  220040700       4           7
     event_id  t0_idx  target_idx
31  220040700      33          36
32  220040700      34          37
33  220040700      35          38
count    34.000000
mean     18.500000
std       9.958246
min       2.000000
25%      10.250000
50%      18.500000
75%      26.750000
max      35.000000
Name: t0_idx, dtype: float64
[ 2  3  4  5  6  7  8  9 10 11]  ... n_unique = 34


In [9]:
from datetime import datetime,timedelta

def parse_event_datetime(event_id: str) -> datetime:
    """
    event_id format: YYDDDHHMM
    Example: '200500700' -> 2020, DOY 050, 07:00
    """
    s = str(event_id)
    if len(s) != 9:
        raise ValueError(f"Expected 9-digit YYDDDHHMM, got '{s}' (len={len(s)})")

    yy   = int(s[0:2])      # 20
    doy  = int(s[2:5])      # 050
    hhmm = s[5:9]           # 0700
    hour = int(hhmm[0:2])   # 07
    minute = int(hhmm[2:4]) # 00

    year = 2000 + yy
    return datetime.strptime(f"{year} {doy} {hour:02d} {minute:02d}", "%Y %j %H %M")


In [10]:
##### Loop over more years ######

# ------------------------------------------------
# -------------------- CONFIG --------------------
# ------------------------------------------------


DATA_DIR = Path("/store_new/mch/msclim/antoumos/R/develop/NOWPRECIP/new_project/radar_data")
FILES = sorted(DATA_DIR.glob("obs.new.*.rda"))

LEAD_STEPS = 3   # +30 min for 10-min timestep
HIST_STEPS = 3   # t0-20, t0-10, t0
STEP = 1         # 10-min stride (use 2 for 20-min stride)

R_ARRAY_KEY = "obs"
# ------------------------------------------------

print(f"Found {len(FILES)} files in {DATA_DIR}")

X_list, Y_list, meta_list = [], [], []
skipped = 0

for idx, f in enumerate(FILES, start=1):
    event_id = f.stem.split(".")[-1]  # obs.new.220040700 -> 220040700
    result = pyreadr.read_r(str(f))
    #arr = result[R_ARRAY_KEY]

    # Choose the array
    if R_ARRAY_KEY is not None:
        if R_ARRAY_KEY not in result:
            print(f"[{idx}/{len(FILES)}] Skip {f.name}: missing key {R_ARRAY_KEY}")
            skipped += 1
            continue
        arr = result[R_ARRAY_KEY]
        key_used = R_ARRAY_KEY
    else:
        # auto-pick: first 3D object (better: set R_ARRAY_KEY explicitly)
        candidates = [(k, v) for k, v in result.items() if hasattr(v, "ndim") and v.ndim == 3]
        if not candidates:
            print(f"[{idx}/{len(FILES)}] Skip {f.name}: no 3D array in keys={list(result.keys())}")
            skipped += 1
            continue
        key_used, arr = candidates[0]

    # Expand samples for this event
    X, Y, meta = expand_event_to_samples_np(
        arr, event_id=event_id, lead_steps=LEAD_STEPS, hist_steps=HIST_STEPS, step=STEP
    )

    if len(meta) == 0:
        print(f"[{idx}/{len(FILES)}] Skip {f.name}: not enough timesteps (arr shape={np.asarray(arr).shape})")
        skipped += 1
        continue
    
    # ---- Add more data artifacts ----
    event_dt = parse_event_datetime(event_id)

    meta["event_datetime"] = event_dt
    meta["year"] = event_dt.year
    meta["season"] = season_from_month(event_dt.month)

    dt_step = timedelta(minutes=10)

    meta["t0_datetime"] = event_dt + meta["t0_idx"] * dt_step
    meta["target_datetime"] = event_dt + meta["target_idx"] * dt_step
    
    X_list.append(X)
    Y_list.append(Y)
    meta_list.append(meta)

    if idx % 25 == 0 or idx == 1:
        print(f"[{idx}/{len(FILES)}] {f.name} key={key_used} arr={np.asarray(arr).shape} -> samples={len(meta)}")



Found 253 files in /store_new/mch/msclim/antoumos/R/develop/NOWPRECIP/new_project/radar_data
34
[1/253] obs.new.200500700.rda key=obs arr=(501, 371, 39) -> samples=34
34
34
34
34
34
34
34
34
34
34
34
34
34
34
34
34
34
34
34
34
34
34
34
34
[25/253] obs.new.202061300.rda key=obs arr=(501, 371, 39) -> samples=34
34
34
34
34
34
34
34
34
34
34
34
34
34
34
34
34
34
34
34
34
34
34
34
34
34
[50/253] obs.new.203411600.rda key=obs arr=(501, 371, 39) -> samples=34
34
34
34
34
34
34
34
34
34
34
34
34
34
34
34
34
34
34
34
34
34
34
34
34
34
[75/253] obs.new.211242200.rda key=obs arr=(501, 371, 39) -> samples=34
34
34
34
34
34
34
34
34
34
34
34
34
34
34
34
34
34
34
34
34
34
34
34
34
34
[100/253] obs.new.211960400.rda key=obs arr=(501, 371, 39) -> samples=34
34
34
34
34
34
34
34
34
34
34
34
34
34
34
34
34
34
34
34
34
34
34
34
34
34
[125/253] obs.new.213611900.rda key=obs arr=(501, 371, 39) -> samples=34
34
34
34
34
34
34
34
34
34
34
34
34
34
34
34
34
34
34
34
34
34
34
34
34
34
[150/253] obs.new.221550

In [18]:
meta_all = pd.concat(meta_list, ignore_index=True) if meta_list else pd.DataFrame()
print("meta_all:", meta_all.shape)
print("Unique events:", meta_all["event_id"].nunique() if not meta_all.empty else 0)
#out_csv = DATA_DIR / "phase2_samples_meta.csv"
#meta_all.to_csv(out_csv, index=False)
#print("Saved:", out_csv)

meta_all: (8602, 8)
Unique events: 253


In [21]:
meta_all["event_datetime"].dtype

dtype('<M8[us]')

In [20]:
meta_all.head(10)

,event_id,t0_idx,target_idx,event_datetime,year,season,t0_datetime,target_datetime
0,200500700,2,5,2020-02-19 07:00:00,2020,DJF,2020-02-19 07:20:00,2020-02-19 07:50:00
1,200500700,3,6,2020-02-19 07:00:00,2020,DJF,2020-02-19 07:30:00,2020-02-19 08:00:00
2,200500700,4,7,2020-02-19 07:00:00,2020,DJF,2020-02-19 07:40:00,2020-02-19 08:10:00
3,200500700,5,8,2020-02-19 07:00:00,2020,DJF,2020-02-19 07:50:00,2020-02-19 08:20:00
4,200500700,6,9,2020-02-19 07:00:00,2020,DJF,2020-02-19 08:00:00,2020-02-19 08:30:00
5,200500700,7,10,2020-02-19 07:00:00,2020,DJF,2020-02-19 08:10:00,2020-02-19 08:40:00
6,200500700,8,11,2020-02-19 07:00:00,2020,DJF,2020-02-19 08:20:00,2020-02-19 08:50:00
7,200500700,9,12,2020-02-19 07:00:00,2020,DJF,2020-02-19 08:30:00,2020-02-19 09:00:00
8,200500700,10,13,2020-02-19 07:00:00,2020,DJF,2020-02-19 08:40:00,2020-02-19 09:10:00
9,200500700,11,14,2020-02-19 07:00:00,2020,DJF,2020-02-19 08:50:00,2020-02-19 09:20:00


In [12]:
# Concatenate
X_all = np.concatenate(X_list, axis=0) if X_list else np.empty((0, HIST_STEPS, 0, 0), dtype=np.float32)
Y_all = np.concatenate(Y_list, axis=0) if Y_list else np.empty((0, 1, 0, 0), dtype=np.float32)

print("\nDONE")
print("Skipped:", skipped)
print("X_all:", X_all.shape)
print("Y_all:", Y_all.shape)

X_all = np.nan_to_num(X_all, nan=0.0)
Y_all = np.nan_to_num(Y_all, nan=0.0)

# Save
#out_npz = DATA_DIR / "phase2_samples.npz"
#np.savez_compressed(out_npz, X=X_all, Y=Y_all)
#print("Saved:", out_npz)


DONE
Skipped: 0
X_all: (8602, 3, 501, 371)
Y_all: (8602, 1, 501, 371)


In [15]:
X_all.shape


(8602, 3, 501, 371)

In [63]:
#### Sanity checks #####

print("NaN fraction X:", np.isnan(X_all).mean())
print("NaN fraction Y:", np.isnan(Y_all).mean())

print("Min/Max X:", X_all.min(), X_all.max())
print("Min/Max Y:", Y_all.min(), Y_all.max())

print(meta_all.groupby("event_id").size().describe())

##### Sanity checks before training ######

### mean and max intensity ####

Y_flat = Y_all.reshape(Y_all.shape[0], -1)

meta_all["mean_intensity"] = Y_flat.mean(axis=1)
meta_all["max_intensity"]  = Y_flat.max(axis=1)

### conditioned on wet ###

wet_mask = Y_flat > 0.1  # or your threshold

meta_all["mean_intensity_wet"] = [
    Y_flat[i][wet_mask[i]].mean() if wet_mask[i].any() else 0
    for i in range(Y_flat.shape[0])
]

meta_all["wet_fraction"] = wet_mask.sum(axis=1) / Y_flat.shape[1]

print(meta_all["mean_intensity_wet"].describe())

print(meta_all["max_intensity"].describe())

NaN fraction X: 0.0
NaN fraction Y: 0.0
Min/Max X: 0.0 20.0
Min/Max Y: 0.0 20.0
count    253.0
mean      34.0
std        0.0
min       34.0
25%       34.0
50%       34.0
75%       34.0
max       34.0
dtype: float64
count    8602.000000
mean        0.367340
std         0.196236
min         0.000000
25%         0.235494
50%         0.309568
75%         0.438656
max         1.455090
Name: mean_intensity_wet, dtype: float64
count    8602.000000
mean        5.950188
std         6.196747
min         0.000000
25%         1.310054
50%         2.975289
75%         8.917533
max        20.000000
Name: max_intensity, dtype: float64


In [61]:
#### Domain coverage ####

for thr in [0.1, 0.5, 1.0, 2.0]:
    wf = (Y_flat > thr).sum(axis=1) / Y_flat.shape[1]
    print(f"\nThreshold {thr} mm / 10 min")
    print(pd.Series(wf).describe())



Threshold 0.1 mm / 10 min
count    8602.000000
mean        0.156016
std         0.091916
min         0.000000
25%         0.088157
50%         0.146567
75%         0.212302
max         0.497232
dtype: float64

Threshold 0.5 mm / 10 min
count    8602.000000
mean        0.029316
std         0.030710
min         0.000000
25%         0.005193
50%         0.018752
75%         0.043673
max         0.180033
dtype: float64

Threshold 1.0 mm / 10 min
count    8602.000000
mean        0.008329
std         0.011839
min         0.000000
25%         0.000194
50%         0.002814
75%         0.012207
max         0.091176
dtype: float64

Threshold 2.0 mm / 10 min
count    8602.000000
mean        0.002254
std         0.004416
min         0.000000
25%         0.000000
50%         0.000202
75%         0.002377
max         0.043363
dtype: float64


In [61]:
#### Class imbalance #####

for thr in [0.1, 0.5, 1.0, 2.0]:
    frac = (Y_flat > thr).sum() / Y_flat.size
    print(f"Pixel fraction > {thr} mm/10min: {frac:.5f}")

Pixel fraction > 0.1 mm/10min: 0.15602
Pixel fraction > 0.5 mm/10min: 0.02932
Pixel fraction > 1.0 mm/10min: 0.00833
Pixel fraction > 2.0 mm/10min: 0.00225


In [68]:
for thr in [0.5, 1.0, 2.0]:
    frac = ((Y_flat > thr).sum(axis=1) > 0).mean()
    print(f"Samples containing > {thr}: {frac:.3f}")

Samples containing > 0.5: 0.971
Samples containing > 1.0: 0.841
Samples containing > 2.0: 0.610


In [ ]:
### Quantiles ####

vals_all = Y_flat.ravel()
vals_all = vals_all[np.isfinite(vals_all)]

quantiles = np.percentile(vals, [50, 75, 90, 95, 99, 99.5, 99.9])
print(quantiles)
thr = 0.1
vals_wet = vals_all[vals_all > thr]
q_wet = np.percentile(vals_wet, [50, 75, 90, 95, 99, 99.5, 99.9])
print(f"WET pixels (> {thr}) quantiles:", q_wet)



[0.         0.02952056 0.185      0.34814638 0.9094514  1.30103874
 3.32666675]
WET pixels (> 0.1) quantiles: [0.23929685 0.41558522 0.71821898 1.03440219 2.49534202 3.91277766
 9.57522321]


In [62]:
### Intensity Frequency Table
bins = [0, 0.1, 0.5, 1.0, 2.0, 5.0, 20.0]
hist, edges = np.histogram(vals, bins=bins)

for i in range(len(hist)):
    frac = hist[i] / len(vals)
    print(f"{edges[i]}–{edges[i+1]}: {frac:.4f}")


NameError: name 'vals' is not defined

In [ ]:
##### Define the operational split ######

# Event Table

events = (
    meta_all[["event_id", "event_datetime", "season"]]
    .drop_duplicates()
)

events = events.sort_values(["season", "event_datetime"])

In [40]:
## Split each season independently

train_events = []
val_events = []
test_events = []

for season, df in events.groupby("season"):

    df = df.sort_values("event_datetime")
    n = len(df)

    train_end = int(0.7 * n)
    val_end   = int(0.85 * n)

    train_events.extend(df.iloc[:train_end]["event_id"])
    val_events.extend(df.iloc[train_end:val_end]["event_id"])
    test_events.extend(df.iloc[val_end:]["event_id"])

In [41]:
#### Tag each sample ####

meta_all["split"] = "train"

meta_all.loc[meta_all["event_id"].isin(val_events), "split"] = "val"
meta_all.loc[meta_all["event_id"].isin(test_events), "split"] = "test"

print(meta_all["split"].value_counts())
print(pd.crosstab(meta_all["split"], meta_all["season"]))

split
train    5984
test     1360
val      1258
Name: count, dtype: int64
season   DJF   JJA   MAM   SON
split                         
test     272   442   374   272
train   1190  1938  1666  1190
val      238   408   340   272


In [ ]:
## Compare summary statistics

meta_all.groupby("split")[["max_intensity",
                           "mean_intensity_wet",
                           "wet_fraction"]].describe()

max_intensity                                                          \
              count      mean       std  min       25%       50%        75%   
split                                                                         
test         1360.0  7.628269  7.210059  0.0  1.596081  3.849213  14.823819   
train        5984.0  5.826715  6.121252  0.0  1.264937  2.936906   8.617070   
val          1258.0  4.723378  4.855294  0.0  1.324363  2.461963   6.615049   

                 mean_intensity_wet            ...                      \
             max              count      mean  ...       75%       max   
split                                          ...                       
test   20.000000             1360.0  0.433818  ...  0.531187  1.446344   
train  19.738037             5984.0  0.355050  ...  0.427008  1.455090   
val    19.738037             1258.0  0.353934  ...  0.436954  1.045181   

      wet_fraction                                                         \
             count      mean       std  min       25%       50%       75%   
split                                                                       
test        1360.0  0.144622  0.090670  0.0  0.082564  0.126911  0.199489   
train       5984.0  0.158527  0.093767  0.0  0.088256  0.148019  0.215892   
val         1258.0  0.156395  0.083075  0.0  0.100738  0.157475  0.208142   

                 
            max  
split            
test   0.466512  
train  0.497232  
val    0.421647  

[3 rows x 24 columns]

In [65]:
bins = [0, 0.1, 0.5, 1, 2, 5, 20]

for split in ["train", "val", "test"]:
    
    idx = meta_all["split"] == split
    vals = Y_flat[idx].ravel()
    
    hist, edges = np.histogram(vals, bins=bins)
    frac = hist / len(vals)
    
    print("\n", split)
    for i in range(len(hist)):
        print(f"{edges[i]}–{edges[i+1]}: {frac[i]:.4f}")


 train
0.0–0.1: 0.8415
0.1–0.5: 0.1303
0.5–1.0: 0.0205
1.0–2.0: 0.0058
2.0–5.0: 0.0016
5.0–20.0: 0.0005

 val
0.0–0.1: 0.8436
0.1–0.5: 0.1255
0.5–1.0: 0.0229
1.0–2.0: 0.0062
2.0–5.0: 0.0015
5.0–20.0: 0.0002

 test
0.0–0.1: 0.8553
0.1–0.5: 0.1121
0.5–1.0: 0.0215
1.0–2.0: 0.0073
2.0–5.0: 0.0027
5.0–20.0: 0.0011


In [71]:
#### logarithmic transformation to reduce skewness ####

X_all_log = np.log1p(X_all)
Y_all_log = np.log1p(Y_all)

X_all_log = X_all_log.astype(np.float32)
Y_all_log = Y_all_log.astype(np.float32)


In [ ]:
##### Baseline construction (optional) ####

test_mask = meta_all["split"] == "test"

X_test = X_all[test_mask]
Y_test = Y_all[test_mask]

Y_persist = X_test[:, -1, :, :]   # last observed frame = t0

mse_persist = np.mean((Y_persist - Y_test)**2)
mae_persist = np.mean(np.abs(Y_persist - Y_test))

print("Persistence MSE:", mse_persist)
print("Persistence MAE:", mae_persist)


MemoryError: Unable to allocate 1.25 TiB for an array with shape (1360, 1360, 501, 371) and data type float32

In [ ]:
### Final dataset ####

train_mask = meta_all["split"] == "train"
val_mask   = meta_all["split"] == "val"
test_mask  = meta_all["split"] == "test"

X_train = X_all_log[train_mask]
Y_train = Y_all_log[train_mask]

X_val = X_all_log[val_mask]
Y_val = Y_all_log[val_mask]

X_test = X_all_log[test_mask]
Y_test = Y_all_log[test_mask]

meta_train = meta_all.loc[train_mask].reset_index(drop=True)
meta_val   = meta_all.loc[val_mask].reset_index(drop=True)
meta_test  = meta_all.loc[test_mask].reset_index(drop=True)

###### Saving (as tensors?) ######



X_train: (5984, 3, 501, 371) Y_train: (5984, 1, 501, 371)
X_val  : (1258, 3, 501, 371) Y_val  : (1258, 1, 501, 371)
X_test : (1360, 3, 501, 371) Y_test : (1360, 1, 501, 371)
